# CP3 — Task 2: Candidate Set & Metrik Evaluasi

**Proyek:** Amazon Reviews 2023 – Musical Instruments (Track G)

**Tujuan notebook ini**
1. Membaca **candidate universe resmi** (`candidate_items_warm.csv`) dari data proyek, bukan membuat kandidat baru.
2. Mengeluarkan item yang sudah diinteraksikan masing-masing pengguna pada **training**.
3. Membuat dan menguji implementasi **Recall@10** dan **NDCG@10** dengan relevansi biner (rating ≥ 4).
4. Memastikan semua rekomendasi memenuhi protokol yang sama (*candidate set*, *evaluable users*, tidak ada item training-seen, maksimal 10 item).
5. Menjalankan **smoke test** dengan Popularity dan menyimpan laporan audit sebagai bukti Task 2.

**Petunjuk:** Jalankan dari **folder utama repository**, seperti notebook Task 1 temanmu. Folder ini harus memiliki `src/`, `MAIN DATA-...`, dan `SUPPORTING_DATA-...` yang sudah diekstrak. Notebook ini **tidak menggunakan test** dan tidak melatih model penelitian baru. Nilai dari Popularity hanya untuk mengecek bahwa alur evaluasi bekerja.

## 1. Import dan muat dataset yang sama dengan Task 1

Notebook temanmu menggunakan fungsi `src.data.load_project_data(PROJECT_ROOT)`. Kita memakai fungsi **yang sama** agar file resmi yang dibaca konsisten.

In [1]:
from pathlib import Path
import sys
import math
import json
import numpy as np
import pandas as pd

# Cari folder repository jika notebook berada di root atau satu tingkat di bawahnya.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. Jalankan notebook dari repository yang berisi src/data.py "
        "dan folder MAIN DATA serta SUPPORTING_DATA hasil ekstraksi."
    )
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import load_project_data

K = 10
POSITIVE_THRESHOLD = 4
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "task2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

project_data = load_project_data(PROJECT_ROOT)
train = project_data.train.copy()
validation_warm = project_data.validation_warm.copy()
items = project_data.items.copy()
candidates = list(dict.fromkeys(str(x) for x in project_data.candidates))
eval_users = list(dict.fromkeys(str(x) for x in project_data.evaluable_users))

for df in [train, validation_warm]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)
items["item_id"] = items["item_id"].astype(str)
train["rating"] = pd.to_numeric(train["rating"], errors="raise")
validation_warm["rating"] = pd.to_numeric(validation_warm["rating"], errors="raise")

print("Repository        :", PROJECT_ROOT)
print("Training rows     :", f"{len(train):,}")
print("Validation warm   :", f"{len(validation_warm):,}")
print("Candidate items   :", f"{len(candidates):,}")
print("Evaluable users   :", f"{len(eval_users):,}")

assert len(train) == 427_957, "Ukuran official train berbeda dengan spesifikasi proyek."
assert candidates and eval_users, "Candidate set/evaluable users kosong."

Repository        : D:\semester 6\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems
Training rows     : 427,957
Validation warm   : 27,537
Candidate items   : 22,753
Evaluable users   : 11,659


## 2. Audit kumpulan kandidat produk

Satu **candidate universe** resmi digunakan oleh semua model. Untuk tiap pengguna, kita hanya mengecualikan item yang **sudah dia interaksikan di training**. Karena itu, daftar akhir per pengguna mungkin sedikit berbeda, tetapi aturan dan candidate universe awalnya tetap sama.

In [2]:
candidate_set = set(candidates)
train_item_set = set(train["item_id"])
metadata_item_set = set(items["item_id"])
eval_user_set = set(eval_users)
train_user_set = set(train["user_id"])

assert len(candidates) == len(candidate_set), "Ada duplikat setelah normalisasi kandidat."
assert candidate_set.issubset(train_item_set), "Ada kandidat yang tidak muncul di training."
assert candidate_set.issubset(metadata_item_set), "Ada kandidat tanpa metadata item."
assert eval_user_set.issubset(train_user_set), "Ada evaluable user yang tidak ada di training."

seen_by_user = {
    uid: set(group["item_id"])
    for uid, group in train.groupby("user_id", sort=False)
}

def allowed_for_user(user_id, item_id):
    """Benar jika produk berada di kandidat resmi dan belum terlihat saat training."""
    uid, iid = str(user_id), str(item_id)
    return (iid in candidate_set) and (iid not in seen_by_user.get(uid, set()))

sample_user = eval_users[0]
seen_sample = seen_by_user.get(sample_user, set())
print("Candidate universe :", len(candidate_set))
print("Training items     :", len(train_item_set))
print("Metadata items     :", len(metadata_item_set))
print("Contoh pengguna    :", sample_user)
print("Pernah diinteraksi :", len(seen_sample), "item training")
print("Kandidat valid     :", len(candidate_set - seen_sample), "item untuk pengguna ini")
print("Audit candidate set: LULUS")

Candidate universe : 22753
Training items     : 22753
Metadata items     : 23582
Contoh pengguna    : AE22B7BMXT7GAGY3XLHICMESNAPA
Pernah diinteraksi : 7 item training
Kandidat valid     : 22746 item untuk pengguna ini
Audit candidate set: LULUS


## 3. Siapkan jawaban yang benar dari validation (*ground truth*)

Suatu produk dianggap relevan jika rating di validation **4 atau 5**, item ada dalam kandidat resmi, dan produk belum diinteraksikan pengguna itu di training. Kita tidak memakai validation untuk melatih atau membuat profil pengguna.

**Catatan:** Jika validation memuat pengulangan pasangan pengguna-item yang juga ada di training, notebook mencatatnya dan mengecualikannya dari target yang memang mustahil direkomendasikan sesuai aturan *seen-item exclusion*. Daftar evaluable users tetap mengikuti file resmi.

In [3]:
valid = validation_warm[
    validation_warm["user_id"].isin(eval_user_set)
    & validation_warm["item_id"].isin(candidate_set)
    & (validation_warm["rating"] >= POSITIVE_THRESHOLD)
].copy()

valid["training_seen"] = [
    iid in seen_by_user.get(uid, set())
    for uid, iid in zip(valid["user_id"], valid["item_id"])
]
n_overlap = int(valid["training_seen"].sum())
relevant_rows = valid.loc[~valid["training_seen"]].copy()
relevant_by_user = {
    uid: set(group["item_id"])
    for uid, group in relevant_rows.groupby("user_id", sort=False)
}
# Semua pengguna resmi dipertahankan dalam perhitungan macro-average.
for uid in eval_users:
    relevant_by_user.setdefault(uid, set())

users_no_target = sum(len(relevant_by_user[uid]) == 0 for uid in eval_users)
print("Positive validation rows, candidate eligible:", len(valid))
print("Positive validation rows yang training-seen:", n_overlap)
print("Positive target rows setelah seen-exclusion:", len(relevant_rows))
print("Evaluable users tanpa target setelah exclusion:", users_no_target)
print("Jumlah pengguna denominator macro-average:", len(eval_users))

assert set(relevant_by_user) == eval_user_set
assert all(set(t).issubset(candidate_set) for t in relevant_by_user.values())
assert all(not (t & seen_by_user.get(u, set())) for u, t in relevant_by_user.items())

Positive validation rows, candidate eligible: 24202
Positive validation rows yang training-seen: 0
Positive target rows setelah seen-exclusion: 24202
Evaluable users tanpa target setelah exclusion: 0
Jumlah pengguna denominator macro-average: 11659


## 4. Fungsi metrik: Recall@10 dan NDCG@10

- **Recall@10:** bagian item relevan yang ditemukan di 10 rekomendasi teratas.
- **NDCG@10:** skor lebih tinggi jika item relevan berada lebih dekat ke urutan pertama.
- **Macro-average:** hitung nilai per pengguna, kemudian rata-ratakan untuk daftar pengguna evaluasi resmi.

Untuk pengguna tanpa item relevan, metrik ditetapkan `0` agar denominator daftar pengguna tetap konsisten; jumlah pengguna seperti ini dilaporkan pada audit.

In [4]:
def recall_at_k(ranked_items, relevant_items, k=10):
    ranked = [str(i) for i in ranked_items[:k]]
    targets = set(map(str, relevant_items))
    if not targets:
        return 0.0
    if len(ranked) != len(set(ranked)):
        raise ValueError("Ranking mengandung item duplikat.")
    return len(set(ranked) & targets) / len(targets)


def ndcg_at_k(ranked_items, relevant_items, k=10):
    ranked = [str(i) for i in ranked_items[:k]]
    targets = set(map(str, relevant_items))
    if not targets:
        return 0.0
    if len(ranked) != len(set(ranked)):
        raise ValueError("Ranking mengandung item duplikat.")
    dcg = sum(1.0 / math.log2(pos + 2) for pos, item in enumerate(ranked) if item in targets)
    ideal_count = min(k, len(targets))
    idcg = sum(1.0 / math.log2(pos + 2) for pos in range(ideal_count))
    return dcg / idcg


def extract_item_ids(ranking):
    """Dukung output ['item1',...] atau [('item1', score), ...] ala notebook Task 1."""
    out = []
    for entry in ranking:
        if isinstance(entry, (tuple, list)):
            out.append(str(entry[0]))
        else:
            out.append(str(entry))
    return out


def audit_recommendations(recs_by_user, model_name="model", k=10):
    """Gagal dengan pesan jelas jika recommender melanggar protokol Task 2."""
    model_users = set(map(str, recs_by_user.keys()))
    if model_users != eval_user_set:
        missing = len(eval_user_set - model_users)
        excess = len(model_users - eval_user_set)
        raise AssertionError(f"{model_name}: pengguna tidak sama (missing={missing}, excess={excess}).")

    violation_counts = {"not_candidate": 0, "already_seen": 0, "duplicate": 0, "too_many": 0}
    for user_id in eval_users:
        ranking = extract_item_ids(recs_by_user[user_id])
        violation_counts["not_candidate"] += sum(item not in candidate_set for item in ranking)
        violation_counts["already_seen"] += sum(item in seen_by_user.get(user_id, set()) for item in ranking)
        violation_counts["duplicate"] += len(ranking) - len(set(ranking))
        violation_counts["too_many"] += int(len(ranking) > k)
    if any(violation_counts.values()):
        raise AssertionError(f"{model_name}: pelanggaran protokol: {violation_counts}")
    print(f"{model_name}: audit lulus untuk {len(eval_users):,} pengguna; kandidat, seen-item, duplikat, Top-{k} OK")
    return violation_counts


def evaluate_topk(recs_by_user, model_name, k=10):
    audit_recommendations(recs_by_user, model_name=model_name, k=k)
    rows = []
    for uid in eval_users:
        ranked = extract_item_ids(recs_by_user[uid])[:k]
        relevant_items = relevant_by_user[uid]
        rows.append({
            "model": model_name,
            "user_id": uid,
            "n_relevant": len(relevant_items),
            "n_recommended": len(ranked),
            f"recall@{k}": recall_at_k(ranked, relevant_items, k),
            f"ndcg@{k}": ndcg_at_k(ranked, relevant_items, k),
        })
    per_user = pd.DataFrame(rows)
    summary = {
        "model": model_name,
        "users_evaluated": len(per_user),
        f"recall@{k}": float(per_user[f"recall@{k}"].mean()),
        f"ndcg@{k}": float(per_user[f"ndcg@{k}"].mean()),
    }
    assert np.isfinite(per_user[[f"recall@{k}", f"ndcg@{k}"]].to_numpy()).all()
    assert per_user[[f"recall@{k}", f"ndcg@{k}"]].ge(0).all().all()
    assert per_user[[f"recall@{k}", f"ndcg@{k}"]].le(1 + 1e-9).all().all()
    return summary, per_user

## 5. Sanity check (uji contoh yang jawabannya sudah diketahui)

Sel ini memastikan rumus benar secara matematis sebelum dipakai pada data asli. Jika ada kesalahan, `assert` akan menghentikan eksekusi.

In [5]:
# Dua item relevan A dan B.
truth = {"A", "B"}

# Keduanya di posisi terbaik -> Recall=1 dan NDCG=1.
assert recall_at_k(["A", "B", "X"], truth, 10) == 1.0
assert math.isclose(ndcg_at_k(["A", "B", "X"], truth, 10), 1.0)

# A dan B ditemukan, tetapi B di posisi 3 -> NDCG di bawah 1.
expected_ndcg = (1 + 1/math.log2(4)) / (1 + 1/math.log2(3))
assert recall_at_k(["A", "X", "B"], truth, 10) == 1.0
assert math.isclose(ndcg_at_k(["A", "X", "B"], truth, 10), expected_ndcg)
assert ndcg_at_k(["A", "X", "B"], truth, 10) < 1

# Hanya satu dari dua relevan ditemukan -> Recall=0.5.
assert recall_at_k(["X", "B", "Y"], truth, 10) == 0.5
assert recall_at_k(["X", "Y"], truth, 10) == 0
assert ndcg_at_k(["X", "Y"], truth, 10) == 0

# Pengguna tanpa target relevan -> 0; tidak boleh lebih dari 10 item.
assert recall_at_k(["A"], set(), 10) == 0
assert ndcg_at_k(["A"], set(), 10) == 0
try:
    recall_at_k(["A", "A"], truth, 10)
except ValueError:
    pass
else:
    raise AssertionError("Duplikat rekomendasi harus terdeteksi.")

print("SEMUA SANITY CHECK Recall@10 dan NDCG@10: LULUS")
print("NDCG untuk ranking [A, X, B]:", round(expected_ndcg, 6))

SEMUA SANITY CHECK Recall@10 dan NDCG@10: LULUS
NDCG untuk ranking [A, X, B]: 0.919721


## 6. Smoke test pada data asli: Popularity Baseline

Kita **tidak perlu menjalankan TF-IDF atau Item-CF yang berat lagi** hanya untuk memvalidasi Task 2. Popularity adalah contoh rekomendasi ringan dari **training saja** untuk menguji alur kandidat, pengguna, dan dua metrik. Model lain nanti cukup memberikan output Top-10 dalam format yang sama.

In [6]:
# Hitung popularitas dari TRAINING saja. Ties diputuskan dengan item_id agar konsisten.
positive_train = train.loc[train["rating"] >= POSITIVE_THRESHOLD]
popular_counts = positive_train.groupby("item_id").size().to_dict()
global_ranking = sorted(candidate_set, key=lambda iid: (-popular_counts.get(iid, 0), iid))

popularity_recs = {}
for uid in eval_users:
    seen = seen_by_user.get(uid, set())
    selected = []
    for iid in global_ranking:
        if iid not in seen:
            selected.append((iid, float(popular_counts.get(iid, 0))))
            if len(selected) == K:
                break
    popularity_recs[uid] = selected

popularity_summary, popularity_per_user = evaluate_topk(popularity_recs, "Popularity", K)
display(pd.DataFrame([popularity_summary]))
print("Top-10 untuk contoh pengguna:", sample_user)
display(pd.DataFrame(popularity_recs[sample_user], columns=["item_id", "score"]))

Popularity: audit lulus untuk 11,659 pengguna; kandidat, seen-item, duplikat, Top-10 OK


,model,users_evaluated,recall@10,ndcg@10
0,Popularity,11659,0.013499,0.009222


Top-10 untuk contoh pengguna: AE22B7BMXT7GAGY3XLHICMESNAPA


,item_id,score
0,B09857JRP2,2175.0
1,B0BCK6L7S5,1441.0
2,B0BSGM6CQ9,1404.0
3,B0BPJ4Q6FJ,1271.0
4,B0BTC9YJ2W,1185.0
5,B004XNK7AI,1119.0
6,B0BHSG5C3G,1031.0
7,B08R5GM6YB,1002.0
8,B0C67HCGBR,917.0
9,B0BPLFP5P6,895.0


## 7. Cek hasil juga menggunakan fungsi evaluasi Task 1 (opsional tetapi disarankan)

Karena Task 1 memakai `src.evaluation.evaluate_ranking`, kita bandingkan hasilnya. Jika fungsi tersedia, dua implementasi seharusnya cocok untuk rekomendasi dan target yang sama. Jika berbeda, jangan langsung menyimpulkan benar: periksa denominator, aturan target, dan tie-breaking.

In [7]:
try:
    from src.evaluation import evaluate_ranking
except ImportError:
    print("src.evaluation.evaluate_ranking tidak tersedia; cek silang dilewati.")
else:
    # target diambil dari ground truth Task 2, yang sudah mempertimbangkan seen-item exclusion.
    project_result = evaluate_ranking(popularity_recs, relevant_by_user, eval_users, k=K)
    print("Implementasi src.evaluation:", project_result)
    print("Implementasi Task 2       :", popularity_summary)
    for metric in ["recall@10", "ndcg@10"]:
        assert math.isclose(project_result[metric], popularity_summary[metric], rel_tol=1e-9, abs_tol=1e-12), (
            f"Nilai {metric} tidak sama; periksa fungsi evaluasi Task 1 dan Task 2."
        )
    assert project_result["users_evaluated"] == len(eval_users)
    print("Pemeriksaan silang dengan src.evaluation: LULUS")

Implementasi src.evaluation: {'users_evaluated': 11659, 'recall@10': 0.013498503732927795, 'ndcg@10': 0.009222040875545052}
Implementasi Task 2       : {'model': 'Popularity', 'users_evaluated': 11659, 'recall@10': 0.013498503732927795, 'ndcg@10': 0.009222040875545052}
Pemeriksaan silang dengan src.evaluation: LULUS


## 8. Simpan hasil untuk bukti CP3 Task 2

File yang dibuat: `outputs/task2/task2_audit_summary.json`, `task2_popularity_per_user.csv`, dan `task2_popularity_results.csv`. Tidak menyentuh data test.

In [8]:
audit_summary = {
    "task": "CP3 - Task 2 Candidate Set & Metrics",
    "dataset": "Amazon Reviews 2023 - Musical Instruments",
    "train_interactions": int(len(train)),
    "validation_warm_interactions": int(len(validation_warm)),
    "candidate_universe_size": int(len(candidate_set)),
    "evaluable_users": int(len(eval_users)),
    "rating_relevant_threshold": POSITIVE_THRESHOLD,
    "metric_k": K,
    "positive_validation_training_seen_excluded": n_overlap,
    "evaluable_users_with_zero_target_after_filter": users_no_target,
    "user_specific_seen_exclusion": True,
    "uses_test_data": False,
    "popularity_smoke_test": popularity_summary,
    "sanity_checks": "PASS",
}

(OUTPUT_DIR / "task2_audit_summary.json").write_text(
    json.dumps(audit_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
popularity_per_user.to_csv(OUTPUT_DIR / "task2_popularity_per_user.csv", index=False)
pd.DataFrame([popularity_summary]).to_csv(OUTPUT_DIR / "task2_popularity_results.csv", index=False)

print("Task 2 selesai. File output:")
for p in sorted(OUTPUT_DIR.glob("task2_*")):
    print("-", p.relative_to(PROJECT_ROOT))

Task 2 selesai. File output:
- outputs\task2\task2_audit_summary.json
- outputs\task2\task2_popularity_per_user.csv
- outputs\task2\task2_popularity_results.csv


## 9. Cara memakai fungsi Task 2 untuk model lain

Ketika anggota lain sudah menghasilkan rekomendasi MF atau Hybrid, gunakan **fungsi evaluasi yang sama**:

```python
# Contoh format (bukan eksekusi):
# mf_recs = {"USER_1": [("ITEM_A", 0.9), ("ITEM_B", 0.8)], ...}
# Hasil harus tersedia untuk SETIAP user dalam eval_users.
# mf_summary, mf_per_user = evaluate_topk(mf_recs, "MF", k=10)
# rrf_summary, rrf_per_user = evaluate_topk(rrf_recs, "Hybrid RRF", k=10)
```

**Catatan perbandingan Task 1:** Notebook temanmu menjalankan Popularity, **Item-CF**, dan TF-IDF CBF; jangan menyebut Item-CF sebagai Matrix Factorization. Untuk Track G, model MF dan hybrid harus dinilai **secara terpisah** ketika implementasinya tersedia.

### Kalimat singkat untuk laporan CP3

> Task 2 menggunakan candidate universe warm-start resmi (`candidate_items_warm.csv`) bagi seluruh model, dengan mengecualikan item yang telah diinteraksikan pengguna selama training. Relevansi ditetapkan dari rating validation warm-start ≥ 4. Evaluasi menggunakan Recall@10 dan NDCG@10 dengan perhitungan rata-rata per pengguna (macro-average). Fungsi evaluasi diuji menggunakan contoh dengan nilai yang sudah diketahui dan diverifikasi melalui baseline Popularity. Data test tidak digunakan.